# Install Libraries and Packages

In [1]:
%pip install python-dotenv pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


# Import Libraries

In [2]:
import os
import glob
import re
import subprocess
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# Creation of .env file to hide security information

#### Archivo .env con las credenciales de la base de datos
#### Por favor cree este archivo en la raíz donde tiene el actual script de python con el siguiente formato:

>DB_USER=xxxx # Cambia esto por tu usuario de PostgreSQL

>DB_HOST=xxxx # Cambia esto por la dirección de tu servidor PostgreSQL (usualmente localhost)

>DB_PORT=xxxx # Cambia esto por el puerto de tu servidor PostgreSQL (usualmente 5432 o 5433)

>DB_NAME=xxxx # Cambia esto por el nombre de tu base de datos PostgreSQL

>DB_PASS=xxxx # Cambia esto por tu contraseña de PostgreSQL

# Connect Local PostgreSQL Database

In [3]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

USER = os.getenv("DB_USER")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
RAW_PASS = os.getenv("DB_PASS")

if not all([USER, HOST, PORT, DB_NAME, RAW_PASS]):
    raise ValueError("Error crítico: No se pudieron importar las credenciales desde el archivo .env")

# Configure paths with raster files

In [4]:
PATH_POSTGRES_BIN = r"C:\Program Files\PostgreSQL\18\bin"
base_path = r"C:\Users\Carlos Andres\Desktop\Geovisor_Rio_Acre"

psql_abs = os.path.join(PATH_POSTGRES_BIN, "psql.exe")
raster_abs = os.path.join(PATH_POSTGRES_BIN, "raster2pgsql.exe")

if not os.path.exists(psql_abs) or not os.path.exists(raster_abs):
    raise FileNotFoundError(f"Error: No se encontraron las herramientas de PostgreSQL en: {PATH_POSTGRES_BIN}")

carpetas_principales = [
    "Conversion_HDF_GIS_Raster_p01",
    "Conversion_HDF_GIS_Raster_p02",
    "Conversion_HDF_GIS_Raster_p03",
    "Conversion_HDF_GIS_Raster_p04",
    "Conversion_HDF_GIS_Raster_p05"
]

planes_config = ["p01", "p02", "p03", "p04", "p05"]
os.environ["PGPASSWORD"] = RAW_PASS

# Insert raster in geodatabase using PostGIS

In [5]:
try:
    print("--- INICIANDO SCRIPT 1: POBLACIÓN DE TABLA MAESTRA ---") 
    for plan in planes_config:
        carpeta_excel = os.path.join(base_path, f"TimeSeries_WSEL_{plan}")
        archivo_busqueda = os.path.join(carpeta_excel, f"*_{plan}.xlsx")
        lista_archivos = glob.glob(archivo_busqueda)
        if not lista_archivos:
            print(f"Advertencia: No se encontró archivo Excel para el {plan} en {carpeta_excel}")
            continue
        ruta_excel = lista_archivos[0]
        print(f"Procesando Excel del {plan.upper()}: {os.path.basename(ruta_excel)}")
        df_excel = pd.read_excel(ruta_excel)
        for idx, row in df_excel.iterrows():
            valor_wsel = float(row['WSEL'])
            plan_str = str(row['Plan']).strip()
            cmd_ins_maestra = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -c "INSERT INTO geovisor_data.nivel_wsel_tm (nivel_wsel, plan) VALUES ({valor_wsel:.4f}, \'{plan_str}\') ON CONFLICT (nivel_wsel, plan) DO NOTHING;"'
            subprocess.run(cmd_ins_maestra, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)     
    print("\n¡La tabla maestra `nivel_wsel_tm` está perfectamente cargada.")
except subprocess.CalledProcessError as e:
    print(f"\n Error en psql:")
    if e.stderr:
        print(e.stderr.decode('utf-8', errors='ignore'))
except Exception as e:
    print(f"Error General: {e}")
finally:
    if "PGPASSWORD" in os.environ:
        del os.environ["PGPASSWORD"]

--- INICIANDO SCRIPT 1: POBLACIÓN DE TABLA MAESTRA ---
Procesando Excel del P01: WSEL_Time_Series_Point_527148.5_8782230.4_p01.xlsx
Procesando Excel del P02: WSEL_Time_Series_Point_527148.5_8782230.4_p02.xlsx
Procesando Excel del P03: WSEL_Time_Series_Point_527148.5_8782230.4_p03.xlsx
Procesando Excel del P04: WSEL_Time_Series_Point_527148.5_8782230.4_p04.xlsx
Procesando Excel del P05: WSEL_Time_Series_Point_527148.5_8782230.4_p05.xlsx

¡La tabla maestra `nivel_wsel_tm` está perfectamente cargada.


In [ ]:
def extraer_fecha(nombre_archivo):
    """Parsea las fechas desde el nombre del archivo raster (ej: WSE_02_19APR2026_00_15_00_p01.tif)"""
    match_ts = re.search(r'(\d{2}[A-Z]{3}\d{4})_(\d{2})_(\d{2})_(\d{2})', nombre_archivo)
    if match_ts:
        try:
            str_fecha = f"{match_ts.group(1)}_{match_ts.group(2)}_{match_ts.group(3)}_{match_ts.group(4)}"
            return datetime.strptime(str_fecha, "%d%b%Y_%H_%M_%S")
        except:
            pass
    return datetime(2026, 4, 19, 0, 0, 0)

def extraer_step_numero(nombre_archivo):
    """Extrae estrictamente el número del paso temporal al inicio (ej: WSE_25... -> 25)"""
    match = re.match(r'^WSE_(\d+)', nombre_archivo)
    if match:
        return int(match.group(1))
    return None

try:
    print("--- INICIANDO SCRIPT 2 (OPTIMIZADO SERIE DE TIEMPO) ---")
    
    # Pre-cargamos los excels en memoria
    excels_en_memoria = {}
    for plan in planes_config:
        carpeta_excel = os.path.join(base_path, f"TimeSeries_WSEL_{plan}")
        archivo_busqueda = os.path.join(carpeta_excel, f"*_{plan}.xlsx")
        lista_archivos = glob.glob(archivo_busqueda)
        if lista_archivos:
            excels_en_memoria[plan] = pd.read_excel(lista_archivos[0])

    ruta_temp_sql = os.path.join(base_path, "temp_raster_insert.sql")

    for cp in carpetas_principales:
        sufijo_plan = cp.split('_')[-1]  # Extrae 'p01', 'p02', etc.
        df_plan_excel = excels_en_memoria.get(sufijo_plan)
        
        if df_plan_excel is None:
            print(f"Saltando carpeta {cp}: No se cargaron datos de referencia en Excel.")
            continue
            
        tabla_destino = f"geovisor_data.manchas_inundacion_wsel_{sufijo_plan}"
        
        # CORRECCIÓN CRÍTICA: Apuntamos EXCLUSIVAMENTE a la serie de tiempo, omitiendo resúmenes estáticos
        subcarpeta_exclusiva = "WSE_Time_Series"
        ruta_busqueda = os.path.join(base_path, cp, subcarpeta_exclusiva, "*.tif")
        
        for ruta_tif in glob.glob(ruta_busqueda):
            nombre_archivo = os.path.basename(ruta_tif)
            
            # Filtro de seguridad por nombre de archivo
            if "Summary" in nombre_archivo or "Max" in nombre_archivo:
                continue
                
            step_num = extraer_step_numero(nombre_archivo)
            
            # Si no tiene número de step o excede las filas del Excel, no se procesa
            if step_num is None or step_num > len(df_plan_excel):
                print(f"Ignorado por inconsistencia o falta de step: {nombre_archivo}")
                continue
                
            fecha_timestamp = extraer_fecha(nombre_archivo).strftime("%Y-%m-%d %H:%M:%S")
            
            # Mapeo exacto: Step 1 -> Índice 0 del DataFrame (Fila 1 del Excel)
            cota_wsel_exacta = float(df_plan_excel.iloc[step_num - 1]['WSEL'])
            
            # 1. Generar la cadena SQL original en memoria usando la bandera -e
            cmd_generar_sql = f'"{raster_abs}" -a -e -s 32719 -f raster_wsel_{sufijo_plan} "{ruta_tif}" {tabla_destino}'
            proc = subprocess.run(cmd_generar_sql, shell=True, capture_output=True, text=True, check=True)
            sql_original = proc.stdout
            
            # 2. Re-ingeniería del comando INSERT para inyección simultánea de metadatos
            columna_rast = f"raster_wsel_{sufijo_plan}"
            columnas_nuevas = f"{columna_rast}, fecha_data_{sufijo_plan}, nivel_wsel_{sufijo_plan}, plan_{sufijo_plan}"
            valores_nuevos = f"\\1', '{fecha_timestamp}', {cota_wsel_exacta:.4f}, '{sufijo_plan}'"
            
            sql_modificado = sql_original.replace(f'"{columna_rast}"', columnas_nuevas)
            sql_modificado = re.sub(r"'(.*?)'\s*\)", valores_nuevos + ")", sql_modificado)
            
            # 3. Escribir el script SQL modificado en un archivo físico temporal
            with open(ruta_temp_sql, "w", encoding="utf-8") as f:
                f.write(sql_modificado)
            
            # 4. Enviar a psql aplicando la inserción atómica directa
            cmd_psql = f'"{psql_abs}" -h {HOST} -U {USER} -d "{DB_NAME}" -f "{ruta_temp_sql}"'
            subprocess.run(cmd_psql, shell=True, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            
            print(f"Anexado Exitoso [{sufijo_plan.upper()}]: {nombre_archivo} -> Step: {step_num} -> Cota WSEL: {cota_wsel_exacta:.4f} m")

    # Limpieza final del archivo de intercambio
    if os.path.exists(ruta_temp_sql):
        os.remove(ruta_temp_sql)
        
    print("\n¡PROCESO DE RÁSTERS COMPLETADO CON ÉXITO! Solo las series de tiempo puras han sido importadas.")

except subprocess.CalledProcessError as e:
    print(f"\n Error crítico en comandos nativos de Postgres:")
    if e.stderr:
        print(e.stderr.decode('utf-8', errors='ignore'))
except Exception as e:
    print(f" Error General de ejecución: {e}")
finally:
    if "PGPASSWORD" in os.environ:
        del os.environ["PGPASSWORD"]

--- INICIANDO SCRIPT 2 (OPTIMIZADO SERIE DE TIEMPO) ---
